# DroneAId — Kaggle CPU Benchmark (Track A Constraint Validation)

**Purpose:** Validate C-A1 (model size ≤ 50 MB) and C-A3 (single-sample inference ≤ 3.0 s) on a Kaggle CPU runtime, which uses **x86_64 Intel Xeon** — the same architecture class as the judges' Intel Core i5 Gen 8 test machine.

**Why Kaggle CPU, not local M-series:** Apple Silicon (ARM64) inference numbers are NOT valid for Track A validation. Intel Xeon at Kaggle typically runs slightly slower per-core than i5-8400, so measurements here are **conservatively pessimistic** — if we pass here, we almost certainly pass on judge hardware.

**Runtime setting (IMPORTANT):**
1. Kaggle → Notebook Settings → Accelerator = **None** (CPU only)
2. Internet = Off (mirrors offline constraint C-A5)

**Before running:** upload your `.onnx` models as a Kaggle Dataset and attach it, then update the `DET_MODEL` / `SEG_MODEL` paths below.

## 1. Environment setup

In [ ]:
!pip install -q onnxruntime==1.18.1 py-cpuinfo==9.0.0 psutil==5.9.8 tabulate==0.9.0
import os, platform, json, time, statistics
import numpy as np
import onnxruntime as ort
import psutil, cpuinfo
from pathlib import Path

In [ ]:
# Lock thread count to match Intel i5 Gen 8 (4 physical cores)
THREADS = 4
os.environ['OMP_NUM_THREADS'] = str(THREADS)
os.environ['MKL_NUM_THREADS'] = str(THREADS)
os.environ['OPENBLAS_NUM_THREADS'] = str(THREADS)

info = cpuinfo.get_cpu_info()
print('CPU        :', info.get('brand_raw', 'unknown'))
print('Arch       :', platform.machine())
print('Phys cores :', psutil.cpu_count(logical=False))
print('Log cores  :', psutil.cpu_count(logical=True))
print('RAM        :', round(psutil.virtual_memory().total / 1024**3, 2), 'GB')
print('Kaggle env :', 'KAGGLE_KERNEL_RUN_TYPE' in os.environ)
assert platform.machine().lower() in ('x86_64', 'amd64'), 'Must run on x86_64 for valid Track A benchmark'

## 2. Model paths — UPDATE THESE

Attach your models dataset via the Kaggle sidebar, then set the paths here.

In [ ]:
# TODO: update to your Kaggle dataset paths
DET_MODEL = '/kaggle/input/droneaid-models/yolov12s_sard.onnx'
SEG_MODEL = '/kaggle/input/droneaid-models/yolov12s_rescuenet.onnx'
DET_IMGSZ = 1280
SEG_IMGSZ = 640
RUNS = 20
WARMUP = 3

C_A1_MAX_MB = 50.0
C_A3_MAX_S = 3.0

## 3. Benchmark helpers

In [ ]:
def size_mb(path):
    return round(Path(path).stat().st_size / 1024**2, 3)

def make_session(path, threads=THREADS):
    so = ort.SessionOptions()
    so.intra_op_num_threads = threads
    so.inter_op_num_threads = 1
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    return ort.InferenceSession(path, sess_options=so, providers=['CPUExecutionProvider'])

def bench_model(path, imgsz, runs=RUNS, warmup=WARMUP):
    sess = make_session(path)
    inp = sess.get_inputs()[0].name
    shape = sess.get_inputs()[0].shape
    if len(shape) == 4 and isinstance(shape[2], int):
        imgsz = shape[2]
    x = np.random.default_rng(42).standard_normal((1, 3, imgsz, imgsz), dtype=np.float32)
    for _ in range(warmup):
        sess.run(None, {inp: x})
    lats = []
    for _ in range(runs):
        t = time.perf_counter()
        sess.run(None, {inp: x})
        lats.append(time.perf_counter() - t)
    return {
        'size_mb': size_mb(path),
        'imgsz': imgsz,
        'min': min(lats), 'p50': statistics.median(lats),
        'p95': statistics.quantiles(lats, n=20)[18] if len(lats) >= 20 else max(lats),
        'max': max(lats), 'mean': statistics.fmean(lats),
        'stdev': statistics.stdev(lats),
        'raw': lats,
    }

## 4. Benchmark each model individually (C-A1 validation)

In [ ]:
det_result = bench_model(DET_MODEL, DET_IMGSZ)
seg_result = bench_model(SEG_MODEL, SEG_IMGSZ)

from tabulate import tabulate
rows = [
    ['Detection (SARD)', det_result['size_mb'], det_result['imgsz'], 
     f"{det_result['p50']*1000:.1f}", f"{det_result['p95']*1000:.1f}", f"{det_result['max']*1000:.1f}"],
    ['Segmentation (RescueNet)', seg_result['size_mb'], seg_result['imgsz'],
     f"{seg_result['p50']*1000:.1f}", f"{seg_result['p95']*1000:.1f}", f"{seg_result['max']*1000:.1f}"],
]
total_size = det_result['size_mb'] + seg_result['size_mb']
rows.append(['TOTAL', round(total_size, 3), '-', '-', '-', '-'])
print(tabulate(rows, headers=['Model', 'Size (MB)', 'imgsz', 'p50 (ms)', 'p95 (ms)', 'max (ms)']))
print()
print(f"C-A1 (total ≤ 50 MB): {'PASS' if total_size <= C_A1_MAX_MB else 'FAIL'} — {total_size:.3f} MB")

## 5. End-to-end pipeline (C-A3 validation)

Measures the full single-frame pipeline: detection → segmentation → optical flow → pose → status → world map.

In [ ]:
import cv2

det_sess = make_session(DET_MODEL)
seg_sess = make_session(SEG_MODEL)
det_inp = det_sess.get_inputs()[0].name
seg_inp = seg_sess.get_inputs()[0].name

rng = np.random.default_rng(123)

def run_pipeline():
    stages = {}
    # Detection
    x = rng.standard_normal((1, 3, DET_IMGSZ, DET_IMGSZ), dtype=np.float32)
    t = time.perf_counter(); det_sess.run(None, {det_inp: x}); stages['detection'] = time.perf_counter() - t
    # Stub: 3 detected bboxes
    bboxes = np.array([[100,100,180,260],[300,150,380,320],[500,400,580,560]], dtype=np.float32)
    # Segmentation
    y = rng.standard_normal((1, 3, SEG_IMGSZ, SEG_IMGSZ), dtype=np.float32)
    t = time.perf_counter(); seg_sess.run(None, {seg_inp: y}); stages['segmentation'] = time.perf_counter() - t
    # Farneback optical flow per bbox crop
    t = time.perf_counter()
    for _ in bboxes:
        prev = rng.integers(0, 255, (96, 96), dtype=np.uint8)
        curr = rng.integers(0, 255, (96, 96), dtype=np.uint8)
        cv2.calcOpticalFlowFarneback(prev, curr, None, 0.5, 3, 15, 3, 5, 1.2, 0)
    stages['optical_flow'] = time.perf_counter() - t
    # Pose (bbox ratio)
    t = time.perf_counter()
    for b in bboxes:
        w = b[2] - b[0]; h = b[3] - b[1]
        _ = 'prone' if w/max(h,1e-6) > 1.0 else 'standing'
    stages['pose'] = time.perf_counter() - t
    # Status classification (stub)
    t = time.perf_counter()
    for _ in bboxes: _ = 'static'
    stages['status'] = time.perf_counter() - t
    # World map update
    t = time.perf_counter()
    buf = {f'v_{i}': {'bbox': b.tolist(), 'gps': (0,0)} for i, b in enumerate(bboxes)}
    stages['world_map'] = time.perf_counter() - t
    return stages

for _ in range(WARMUP): run_pipeline()

all_stages = []
for _ in range(RUNS):
    all_stages.append(run_pipeline())

totals = [sum(s.values()) for s in all_stages]
stage_avg = {k: statistics.fmean(s[k] for s in all_stages) for k in all_stages[0]}

print('Stage averages (ms):')
for k, v in stage_avg.items():
    print(f'  {k:<18s} {v*1000:>8.2f}')
print()
print(f'End-to-end p50 : {statistics.median(totals)*1000:.2f} ms')
p95 = statistics.quantiles(totals, n=20)[18] if len(totals) >= 20 else max(totals)
print(f'End-to-end p95 : {p95*1000:.2f} ms')
print(f'End-to-end max : {max(totals)*1000:.2f} ms')
print()
print(f'C-A3 (p95 ≤ 3.0 s): {"PASS" if p95 <= C_A3_MAX_S else "FAIL"}')

## 6. Save report (for Bab 3 proposal)

In [ ]:
report = {
    'system': {
        'cpu': info.get('brand_raw'),
        'arch': platform.machine(),
        'cores_phys': psutil.cpu_count(logical=False),
        'cores_log': psutil.cpu_count(logical=True),
        'ram_gb': round(psutil.virtual_memory().total / 1024**3, 2),
        'kaggle': 'KAGGLE_KERNEL_RUN_TYPE' in os.environ,
    },
    'config': {'threads': THREADS, 'runs': RUNS, 'warmup': WARMUP,
               'det_imgsz': DET_IMGSZ, 'seg_imgsz': SEG_IMGSZ},
    'detection': det_result,
    'segmentation': seg_result,
    'pipeline': {
        'stage_avg_s': stage_avg,
        'total_p50_s': statistics.median(totals),
        'total_p95_s': p95,
        'total_max_s': max(totals),
        'raw_totals_s': totals,
    },
    'constraints': {
        'c_a1_total_size_mb': round(det_result['size_mb'] + seg_result['size_mb'], 3),
        'c_a1_pass': (det_result['size_mb'] + seg_result['size_mb']) <= C_A1_MAX_MB,
        'c_a3_p95_s': p95,
        'c_a3_pass': p95 <= C_A3_MAX_S,
    }
}
Path('/kaggle/working/benchmark_report.json').write_text(json.dumps(report, indent=2))
print('Saved: /kaggle/working/benchmark_report.json')